<a href="https://colab.research.google.com/github/karquiza/task-delay-risk-predictor/blob/main/notebooks/baseline_notebook_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Baseline Model notebook

1. Load the split data
3. Prepare the data
4. Apply the 3 rule checklist and record scores

## The 3 Rules:
**R1. Large task:**

Flag the task if story points are 8 or higher. This is the top quarter of RSF tasks: 15 of 62.

**R2. External dependency:**

Flag the task if it depends on a role outside the core team, including SP Sponsor, PE Privacy/Ethics officer, or SM Store managers, or if the task is an approval or access request.

**R3. High workload:**

Flag the task if any role assigned to it has more than 3 other open tasks across the same date range.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
import numpy as np
import pandas as pd

# Folder in My Drive
BASE = "/content/drive/MyDrive/MAI104 Group Project"
DATA = os.path.join(BASE, "data", "Synthetic_Data_V3.csv")
SPLITS = os.path.join(BASE, "data", "splits")
RESULTS = os.path.join(BASE, "results")
SEEDS = [26, 10, 16, 21, 30, 42, 50, 55, 101, 201]   # 26 = main result

print(os.listdir(BASE))   # quick check that the folder is found

df = pd.read_csv(DATA)
df["Late"] = (df["Risk_Outcome"] != "On_Time").astype(int)
print(df.shape, "rows,", df["Task_ID"].nunique(), "tasks, late share", round(df["Late"].mean(), 3))

['data', 'results']
(1340, 22) rows, 67 tasks, late share 0.274


In [3]:
def checklist_flags(data, workload_threshold):
    """3-rule checklist: story points >= 8, OR external dependency, OR workload >= threshold."""
    return ((data["Story_Points"] >= 8)
            | (data["External_Dependency"] == 1)
            | (data["Assignee_Active_Workload_SP"] >= workload_threshold)).astype(int)


def top20_late_share(data, flags):
    """Late share among the top 20% ranked by the checklist:
    flagged tasks first, ties broken by story points (largest first)."""
    ranked = data.assign(_flag=flags.values).sort_values(
        ["_flag", "Story_Points"], ascending=False, kind="stable")
    k = round(0.2 * len(ranked))
    return ranked["Late"].iloc[:k].mean()

In [4]:
# Selection stage: workload threshold from TRAINING tasks, scored on VALIDATION.
# The test set is not used in this notebook.
rows = []
for seed in SEEDS:
    split = pd.read_csv(os.path.join(SPLITS, f"split_assignments_seed{seed}.csv"))
    data = df.merge(split, on="Task_ID")
    train = data[data["Set"] == "train"]
    val = data[data["Set"] == "validation"]

    threshold = train["Assignee_Active_Workload_SP"].quantile(0.75)
    flags = checklist_flags(val, threshold)

    tp = int(((flags == 1) & (val["Late"] == 1)).sum())
    rows.append({
        "seed": seed,
        "val_tasks": val["Task_ID"].nunique(),
        "val_late_share": round(val["Late"].mean(), 3),
        "workload_threshold": round(threshold, 1),
        "flagged_share": round(flags.mean(), 3),
        "recall": round(tp / val["Late"].sum(), 3),
        "precision": round(tp / flags.sum(), 3),
        "top20_late_share": round(top20_late_share(val, flags), 3),
    })

results = pd.DataFrame(rows)
print(results.to_string(index=False))
print("\nMean over 10 seeds:")
print(results.drop(columns=["seed", "val_tasks", "workload_threshold"]).mean().round(3).to_string())

 seed  val_tasks  val_late_share  workload_threshold  flagged_share  recall  precision  top20_late_share
   26         13           0.215                34.5          0.412   0.500      0.262             0.115
   10         13           0.323                33.0          0.608   0.690      0.367             0.308
   16         13           0.331                33.0          0.854   0.907      0.351             0.404
   21         13           0.223                34.5          0.531   0.741      0.312             0.462
   30         13           0.265                34.1          0.488   0.536      0.291             0.365
   42         13           0.200                36.1          0.373   0.442      0.237             0.269
   50         13           0.154                34.5          0.500   0.650      0.200             0.192
   55         13           0.327                35.0          0.573   0.671      0.383             0.231
  101         13           0.238                35.5   

In [5]:
os.makedirs(RESULTS, exist_ok=True)
results.to_csv(os.path.join(RESULTS, "checklist_baseline_validation.csv"), index=False)
print("Saved to", RESULTS)

Saved to /content/drive/MyDrive/MAI104 Group Project/results
